In [1]:
"""
PIPELINE COMPLETO POR CUENCA — Paso 2+3+4 — Sao Francisco.

Combina en un solo script, para UNA cuenca, los 3 escenarios
(historical, ssp126, ssp585):

  Paso 2+3: remuestreo de poblacion (WorldPop) a la grilla exacta de la
            caracterizacion de sequia, por suma en bloques. Se calcula
            UNA VEZ por año de poblacion (2015 y 2030), no por escenario
            (ssp126 y ssp585 comparten la poblacion de 2030). Si el
            archivo regrideado ya existe en disco, se reutiliza (no se
            recalcula) — ya fue validado con 0.0000% de diferencia.

  Paso 4:   calculo de exposicion = caracteristica_sequia x poblacion,
            para las 3 variables (frequency, duration, severity), para
            cada uno de los 3 escenarios.

Mapeo escenario -> poblacion:
  historical -> WorldPop 2015
  ssp126     -> WorldPop 2030
  ssp585     -> WorldPop 2030

NO genera graficos ni mapas (eso se hace en scripts separados, despues).

Requiere: rasterio, xarray, numpy, pandas
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
import xarray as xr


# ====================================================================
# CONFIG
# ====================================================================

BASIN = "sao_francisco"
BASIN_FOLDER = "Sao_Francisco"  # nombre de carpeta en SPEI_R (con mayusculas)

# Mosaicos de poblacion (Paso 1, YA GENERADOS, no se repiten)
POP_TIF_100M = {
    "2015": "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/4.Processing_outputs/worldpop_under18_2015_SA_100m.tif",
    "2030": "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/4.Processing_outputs/worldpop_under18_2030_SA_100m.tif",
}

# Archivos de caracterizacion de sequia (ensemble) por escenario
DROUGHT_BASE = f"/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.General_anual/2.Outputs/{BASIN_FOLDER}"

SCENARIOS = {
    "historical": {
        "drought_nc": f"{DROUGHT_BASE}/historical/drought_annual_{BASIN}_historical_ensemble.nc",
        "pop_year": "2015",
    },
    "ssp126": {
        "drought_nc": f"{DROUGHT_BASE}/ssp126/drought_annual_{BASIN}_ssp126_ensemble.nc",
        "pop_year": "2030",
    },
    "ssp585": {
        "drought_nc": f"{DROUGHT_BASE}/ssp585/drought_annual_{BASIN}_ssp585_ensemble.nc",
        "pop_year": "2030",
    },
}

POP_OUT_DIR = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/5.Population_regridded/"
EXPOSURE_OUT_DIR = f"/data/brussel/vo/000/bvo00033/vsc11346/Exposure/6.Final_outputs/{BASIN_FOLDER}/"
os.makedirs(POP_OUT_DIR, exist_ok=True)
os.makedirs(EXPOSURE_OUT_DIR, exist_ok=True)

POP_NODATA_FALLBACK = -99999.0
NODATA = -9999.0
BLOCK = 4096

VARS_SEQUIA = ["frequency", "duration", "severity"]

FORCE_RECALC_POP = False  # poner True si por algun motivo hay que forzar el recalculo de poblacion


# ====================================================================
# PASO 2+3 — FUNCIONES (remuestreo de poblacion, suma por bloques)
# ====================================================================

def get_target_grid(nc_path):
    ds = xr.open_dataset(nc_path)
    lat = ds["lat"].values.astype("float64")
    lon = ds["lon"].values.astype("float64")
    ds.close()

    res_lat = float(np.round(np.abs(np.diff(lat)).mean(), 8))
    res_lon = float(np.round(np.abs(np.diff(lon)).mean(), 8))

    west = float(lon.min() - res_lon / 2)
    north = float(lat.max() + res_lat / 2)

    n_rows = len(lat)
    n_cols = len(lon)

    transform = from_origin(west, north, res_lon, res_lat)

    return transform, (n_rows, n_cols), lat, lon, res_lat, res_lon


def regrid_population_blocksum(pop_tif_path, target_transform, target_shape, block=BLOCK):
    n_rows, n_cols = target_shape
    dst_data = np.zeros((n_rows, n_cols), dtype="float64")
    total_sum = 0.0

    with rasterio.open(pop_tif_path) as src:
        src_nodata = src.nodata if src.nodata is not None else POP_NODATA_FALLBACK
        src_transform = src.transform

        t_west = target_transform.c
        t_north = target_transform.f
        t_res_x = target_transform.a
        t_res_y = -target_transform.e
        t_east = t_west + n_cols * t_res_x
        t_south = t_north - n_rows * t_res_y

        full_window = rasterio.windows.from_bounds(t_west, t_south, t_east, t_north, transform=src_transform)
        full_window = full_window.round_offsets().round_lengths()
        full_window = full_window.intersection(Window(0, 0, src.width, src.height))

        row_off0, col_off0 = int(full_window.row_off), int(full_window.col_off)
        n_rows_w, n_cols_w = int(full_window.height), int(full_window.width)

        for r0 in range(0, n_rows_w, block):
            h = min(block, n_rows_w - r0)
            for c0 in range(0, n_cols_w, block):
                w = min(block, n_cols_w - c0)

                win = Window(col_off0 + c0, row_off0 + r0, w, h)
                data = src.read(1, window=win)
                win_transform = src.window_transform(win)

                valid = data != src_nodata
                if not valid.any():
                    continue

                rows_idx, cols_idx = np.nonzero(valid)
                values = data[rows_idx, cols_idx].astype("float64")

                xs = win_transform.c + (cols_idx + 0.5) * win_transform.a
                ys = win_transform.f + (rows_idx + 0.5) * win_transform.e

                dst_col = np.floor((xs - t_west) / t_res_x).astype("int64")
                dst_row = np.floor((t_north - ys) / t_res_y).astype("int64")

                in_bounds = (dst_col >= 0) & (dst_col < n_cols) & (dst_row >= 0) & (dst_row < n_rows)
                if not in_bounds.any():
                    continue

                dst_col = dst_col[in_bounds]
                dst_row = dst_row[in_bounds]
                values_in = values[in_bounds]

                flat_idx = dst_row * n_cols + dst_col
                sums = np.bincount(flat_idx, weights=values_in, minlength=n_rows * n_cols)
                dst_data += sums.reshape(n_rows, n_cols)

                total_sum += float(values_in.sum())

    return dst_data.astype("float32"), total_sum


def sum_source_in_bbox_chunked(pop_tif_path, west, south, east, north, block=BLOCK):
    total = 0.0
    with rasterio.open(pop_tif_path) as src:
        nodata = src.nodata if src.nodata is not None else POP_NODATA_FALLBACK
        full_window = rasterio.windows.from_bounds(west, south, east, north, transform=src.transform)
        full_window = full_window.round_offsets().round_lengths()
        full_window = full_window.intersection(Window(0, 0, src.width, src.height))

        row_off0, col_off0 = int(full_window.row_off), int(full_window.col_off)
        n_rows_w, n_cols_w = int(full_window.height), int(full_window.width)

        for r0 in range(0, n_rows_w, block):
            h = min(block, n_rows_w - r0)
            for c0 in range(0, n_cols_w, block):
                w = min(block, n_cols_w - c0)
                win = Window(col_off0 + c0, row_off0 + r0, w, h)
                data = src.read(1, window=win)
                valid = data != nodata
                total += float(data[valid].sum())
    return total


def save_population_nc(dst_data, lat_target, lon_target, out_path, year, basin):
    dst_data_ascending = dst_data[::-1, :]
    da = xr.DataArray(
        dst_data_ascending, dims=("lat", "lon"),
        coords={"lat": lat_target, "lon": lon_target},
        name="population_under18",
        attrs={
            "long_name": f"Population under 18 ({year}) resampled to drought grid",
            "units": "children per pixel",
            "source": "WorldPop, aggregated by sum (block-wise custom resampling)",
            "_FillValue": NODATA,
            "basin": basin,
        },
    )
    ds_out = da.to_dataset()
    ds_out.attrs["description"] = f"Static population <18, year {year}, regridded to {basin} drought grid"
    ds_out.to_netcdf(out_path)


def get_or_create_population(pop_year, target_transform, target_shape, lat_target, lon_target,
                              west, south, east, north):
    """Reutiliza el .nc de poblacion si ya existe; si no, lo calcula y valida."""
    out_path = f"{POP_OUT_DIR}worldpop_under18_{pop_year}_{BASIN}.nc"

    if os.path.exists(out_path) and not FORCE_RECALC_POP:
        print(f"  Poblacion {pop_year}: ya existe, reutilizando -> {out_path}")
        return out_path

    print(f"  Poblacion {pop_year}: calculando (no existia)...")
    dst_data, suma_post = regrid_population_blocksum(POP_TIF_100M[pop_year], target_transform, target_shape)
    suma_ref = sum_source_in_bbox_chunked(POP_TIF_100M[pop_year], west, south, east, north)
    diff_pct = 100 * (suma_post - suma_ref) / suma_ref if suma_ref != 0 else float("nan")
    print(f"    Suma regrid: {suma_post:,.1f} | Suma ref: {suma_ref:,.1f} | Diff: {diff_pct:.4f}%")
    if abs(diff_pct) > 0.5:
        print("    ATENCION: diferencia de conservacion de masa > 0.5% -- revisar.")

    save_population_nc(dst_data, lat_target, lon_target, out_path, pop_year, BASIN)
    print(f"    Guardado: {out_path}")
    return out_path


# ====================================================================
# PASO 4 — CALCULO DE EXPOSICION
# ====================================================================

def calcular_exposicion(drought_nc, pop_nc, scenario, pop_year):
    ds_drought = xr.open_dataset(drought_nc)
    ds_pop = xr.open_dataset(pop_nc)

    lat_ok = np.array_equal(ds_drought["lat"].values, ds_pop["lat"].values)
    lon_ok = np.array_equal(ds_drought["lon"].values, ds_pop["lon"].values)
    if not (lat_ok and lon_ok):
        raise ValueError(
            f"[{scenario}] Las grillas de sequia y poblacion NO coinciden exactamente. "
            "No se puede multiplicar con seguridad."
        )

    pop_da = ds_pop["population_under18"]
    pop_valid = pop_da.notnull()

    exposure_vars = {}
    for var in VARS_SEQUIA:
        drought_da = ds_drought[var]
        drought_valid = drought_da.notnull()
        valid_mask = drought_valid & pop_valid

        exposure = drought_da * pop_da
        exposure = xr.where(valid_mask, exposure, np.nan)
        exposure.name = f"exposure_{var}"
        exposure.attrs = {
            "long_name": f"Children under 18 exposed, weighted by {var}",
            "formula": f"{var}(pixel,year) x population_under18(pixel)",
            "units": f"children x {var}_units",
            "basin": BASIN,
            "scenario": scenario,
            "model": "ensemble",
            "population_year": pop_year,
        }
        exposure_vars[f"exposure_{var}"] = exposure

        n_valid = int(valid_mask.sum())
        n_total = valid_mask.size
        print(f"    exposure_{var}: {n_valid}/{n_total} pixeles-año validos ({100*n_valid/n_total:.1f}%)")

    ds_exposure = xr.Dataset(exposure_vars, coords={
        "year": ds_drought["year"], "lat": ds_drought["lat"], "lon": ds_drought["lon"],
    })
    ds_exposure.attrs = {
        "description": f"Annual child drought exposure — {BASIN}, {scenario}, ensemble",
        "basin": BASIN, "scenario": scenario, "model": "ensemble",
        "population_source": f"WorldPop {pop_year} (static)",
    }

    out_nc = f"{EXPOSURE_OUT_DIR}exposure_annual_{BASIN}_{scenario}_ensemble.nc"
    encoding = {v: {"_FillValue": NODATA} for v in exposure_vars.keys()}
    ds_exposure.to_netcdf(out_nc, encoding=encoding)
    print(f"    Guardado: {out_nc}")

    # resumen anual (CSV, sin graficos)
    years = ds_drought["year"].values
    resumen = {"year": years}
    for var in VARS_SEQUIA:
        da = ds_exposure[f"exposure_{var}"]
        serie = da.sum(dim=["lat", "lon"], skipna=True).values
        resumen[f"exposure_{var}"] = serie
        print(f"    exposure_{var}: promedio={np.nanmean(serie):,.1f} | "
              f"max={np.nanmax(serie):,.1f} (año {years[np.nanargmax(serie)]}) | "
              f"min={np.nanmin(serie):,.1f} (año {years[np.nanargmin(serie)]})")

    df_resumen = pd.DataFrame(resumen)
    csv_path = f"{EXPOSURE_OUT_DIR}resumen_anual_{BASIN}_{scenario}_ensemble.csv"
    df_resumen.to_csv(csv_path, index=False)
    print(f"    CSV resumen: {csv_path}")

    return out_nc, csv_path


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":

    print(f"{'='*70}\nPIPELINE — {BASIN}\n{'='*70}")

    # -------- Paso 2+3: poblacion (una vez por año, reutilizada entre escenarios) --------
    print("\n--- PASO 2+3: Poblacion ---")

    # grilla objetivo: se toma del historico y se verifica que ssp126/ssp585 compartan la misma
    target_transform, target_shape, lat_target, lon_target, res_lat, res_lon = get_target_grid(
        SCENARIOS["historical"]["drought_nc"]
    )
    n_rows, n_cols = target_shape
    west = target_transform.c
    north = target_transform.f
    east = west + n_cols * res_lon
    south = north - n_rows * res_lat
    print(f"Grilla objetivo (desde historical): shape={target_shape}, "
          f"bbox=({west:.4f},{south:.4f},{east:.4f},{north:.4f})")

    for scenario, cfg in SCENARIOS.items():
        t2, s2, lat2, lon2, _, _ = get_target_grid(cfg["drought_nc"])
        same = np.array_equal(lat_target, lat2) and np.array_equal(lon_target, lon2)
        print(f"  Grilla {scenario} == grilla historical: {same}")
        if not same:
            raise ValueError(
                f"La grilla de {scenario} NO coincide con la de historical. "
                "Revisar antes de continuar (no se puede reutilizar la poblacion regrideada)."
            )

    pop_nc_paths = {}
    for pop_year in ["2015", "2030"]:
        pop_nc_paths[pop_year] = get_or_create_population(
            pop_year, target_transform, target_shape, lat_target, lon_target, west, south, east, north
        )

    # -------- Paso 4: exposicion por escenario --------
    print("\n--- PASO 4: Exposicion ---")

    outputs = []
    for scenario, cfg in SCENARIOS.items():
        print(f"\n  Escenario: {scenario} (poblacion {cfg['pop_year']})")
        out_nc, csv_path = calcular_exposicion(
            cfg["drought_nc"], pop_nc_paths[cfg["pop_year"]], scenario, cfg["pop_year"]
        )
        outputs.append((scenario, out_nc, csv_path))

    print(f"\n\n{'='*70}\nRESUMEN DE ARCHIVOS GENERADOS\n{'='*70}")
    print("Poblacion regrideada:")
    for y, p in pop_nc_paths.items():
        print(f"  {y}: {p}")
    print("\nExposicion por escenario:")
    for scenario, out_nc, csv_path in outputs:
        print(f"  {scenario}:")
        print(f"    {out_nc}")
        print(f"    {csv_path}")

PIPELINE — sao_francisco

--- PASO 2+3: Poblacion ---
Grilla objetivo (desde historical): shape=(40, 25), bbox=(-47.6667,-23.7500,-35.1667,-3.7500)
  Grilla historical == grilla historical: True
  Grilla ssp126 == grilla historical: True
  Grilla ssp585 == grilla historical: True
  Poblacion 2015: calculando (no existia)...
    Suma regrid: 31,165,488.4 | Suma ref: 31,165,488.1 | Diff: 0.0000%
    Guardado: /data/brussel/vo/000/bvo00033/vsc11346/Exposure/5.Population_regridded/worldpop_under18_2015_sao_francisco.nc
  Poblacion 2030: calculando (no existia)...
    Suma regrid: 25,706,679.6 | Suma ref: 25,706,679.5 | Diff: 0.0000%
    Guardado: /data/brussel/vo/000/bvo00033/vsc11346/Exposure/5.Population_regridded/worldpop_under18_2030_sao_francisco.nc

--- PASO 4: Exposicion ---

  Escenario: historical (poblacion 2015)
    exposure_frequency: 16680/30000 pixeles-año validos (55.6%)
    exposure_duration: 16680/30000 pixeles-año validos (55.6%)
    exposure_severity: 16680/30000 pixeles

In [1]:
import xarray as xr
import numpy as np

POP_DIR = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/Approach1/3.Population_regridded/"

archivos = {
    "2015": POP_DIR + "worldpop_under18_2015_sao_francisco.nc",
    "2030": POP_DIR + "worldpop_under18_2030_sao_francisco.nc",
}

datos = {}

for year, path in archivos.items():
    print(f"\n{'='*60}\n{year}\n{'='*60}")
    ds = xr.open_dataset(path)
    print(ds)

    da = ds["population_under18"]
    print(f"\nShape: {da.shape}")
    print(f"Lat range: {float(da.lat.min()):.4f} a {float(da.lat.max()):.4f}")
    print(f"Lon range: {float(da.lon.min()):.4f} a {float(da.lon.max()):.4f}")

    valores = da.values
    n_validos = int(np.isfinite(valores).sum())
    n_total = valores.size
    print(f"Píxeles válidos: {n_validos} de {n_total} ({100*n_validos/n_total:.1f}%)")
    print(f"Min: {np.nanmin(valores):.4f}  |  Max: {np.nanmax(valores):.4f}")
    print(f"Suma total (población <18 en la cuenca): {np.nansum(valores):,.1f}")

    n_negativos = int((valores[np.isfinite(valores)] < 0).sum())
    print(f"Píxeles negativos: {n_negativos}")

    datos[year] = {"lat": da.lat.values, "lon": da.lon.values, "sum": np.nansum(valores),
                    "mask_valida": np.isfinite(valores)}
    ds.close()

# Consistencia entre 2015 y 2030
print(f"\n{'='*60}\nConsistencia 2015 vs 2030\n{'='*60}")
print(f"Misma lat: {np.array_equal(datos['2015']['lat'], datos['2030']['lat'])}")
print(f"Misma lon: {np.array_equal(datos['2015']['lon'], datos['2030']['lon'])}")
print(f"Misma máscara de píxeles válidos: {np.array_equal(datos['2015']['mask_valida'], datos['2030']['mask_valida'])}")

caida_pct = 100 * (datos['2030']['sum'] - datos['2015']['sum']) / datos['2015']['sum']
print(f"\nCambio en población <18 (2015->2030): {caida_pct:+.2f}%")


2015
<xarray.Dataset> Size: 5kB
Dimensions:             (lat: 40, lon: 25)
Coordinates:
  * lat                 (lat) float64 320B -23.5 -23.0 -22.5 ... -5.0 -4.5 -4.0
  * lon                 (lon) float64 200B -47.42 -46.92 ... -35.92 -35.42
Data variables:
    population_under18  (lat, lon) float32 4kB ...
Attributes:
    description:  Static population <18, year 2015, regridded to sao_francisc...

Shape: (40, 25)
Lat range: -23.5000 a -4.0000
Lon range: -47.4167 a -35.4167
Píxeles válidos: 1000 de 1000 (100.0%)
Min: 0.0000  |  Max: 3050783.5000
Suma total (población <18 en la cuenca): 31,165,488.0
Píxeles negativos: 0

2030
<xarray.Dataset> Size: 5kB
Dimensions:             (lat: 40, lon: 25)
Coordinates:
  * lat                 (lat) float64 320B -23.5 -23.0 -22.5 ... -5.0 -4.5 -4.0
  * lon                 (lon) float64 200B -47.42 -46.92 ... -35.92 -35.42
Data variables:
    population_under18  (lat, lon) float32 4kB ...
Attributes:
    description:  Static population <18, year 2

In [2]:
import xarray as xr
import pandas as pd
import numpy as np

BASIN = "sao_francisco"
BASIN_FOLDER = "Sao_Francisco"
EXPOSURE_DIR = f"/data/brussel/vo/000/bvo00033/vsc11346/Exposure/Approach1/4.Final_outputs/{BASIN_FOLDER}/"

VARS_SEQUIA = ["frequency", "duration", "severity"]
ESCENARIOS = ["historical", "ssp126", "ssp585"]

for scenario in ESCENARIOS:
    print(f"\n{'='*70}\n{scenario}\n{'='*70}")

    nc_path = EXPOSURE_DIR + f"exposure_annual_{BASIN}_{scenario}_ensemble.nc"
    ds = xr.open_dataset(nc_path)
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Años: {int(ds.year.min())}–{int(ds.year.max())} (n={ds.year.size})")

    for var in VARS_SEQUIA:
        da = ds[f"exposure_{var}"]
        valores = da.values
        n_validos = int(np.isfinite(valores).sum())
        n_total = valores.size
        n_negativos = int((valores[np.isfinite(valores)] < 0).sum())

        print(f"\n  exposure_{var}:")
        print(f"    Píxeles-año válidos: {n_validos}/{n_total} ({100*n_validos/n_total:.1f}%)")
        print(f"    Min: {np.nanmin(valores):,.2f}  |  Max: {np.nanmax(valores):,.2f}")
        print(f"    Negativos: {n_negativos}" + ("  <-- REVISAR" if n_negativos > 0 else ""))

    # Cruce con el CSV resumen
    csv_path = EXPOSURE_DIR + f"resumen_anual_{BASIN}_{scenario}_ensemble.csv"
    df = pd.read_csv(csv_path)
    print(f"\n  CSV resumen ({csv_path.split('/')[-1]}):")
    print(df.describe().to_string())

    # Chequeo cruzado: la suma espacial del NetCDF debe coincidir con el CSV, año por año
    for var in VARS_SEQUIA:
        serie_nc = ds[f"exposure_{var}"].sum(dim=["lat", "lon"], skipna=True).values
        serie_csv = df[f"exposure_{var}"].values
        coincide = np.allclose(serie_nc, serie_csv, rtol=1e-4, equal_nan=True)
        print(f"    exposure_{var}: NetCDF y CSV coinciden -> {coincide}")

    ds.close()

# Comparación rápida entre escenarios (promedio de toda la serie)
print(f"\n{'='*70}\nComparación entre escenarios (promedio de la serie completa)\n{'='*70}")
for var in VARS_SEQUIA:
    print(f"\n  exposure_{var}:")
    for scenario in ESCENARIOS:
        csv_path = EXPOSURE_DIR + f"resumen_anual_{BASIN}_{scenario}_ensemble.csv"
        df = pd.read_csv(csv_path)
        print(f"    {scenario}: promedio={df[f'exposure_{var}'].mean():,.1f}")


historical
Variables: ['exposure_frequency', 'exposure_duration', 'exposure_severity']
Años: 1985–2014 (n=30)

  exposure_frequency:
    Píxeles-año válidos: 16680/30000 (55.6%)
    Min: 0.00  |  Max: 3,050,783.50
    Negativos: 0

  exposure_duration:
    Píxeles-año válidos: 16680/30000 (55.6%)
    Min: 0.00  |  Max: 36,609,400.00
    Negativos: 0

  exposure_severity:
    Píxeles-año válidos: 16680/30000 (55.6%)
    Min: 0.00  |  Max: 51,366,216.00
    Negativos: 0

  CSV resumen (resumen_anual_sao_francisco_historical_ensemble.csv):
              year  exposure_frequency  exposure_duration  exposure_severity
count    30.000000        3.000000e+01       3.000000e+01       3.000000e+01
mean   1999.500000        1.200879e+07       8.279129e+07       8.052859e+07
std       8.803408        6.297291e+06       6.692466e+07       8.030513e+07
min    1985.000000        0.000000e+00       0.000000e+00       0.000000e+00
25%    1992.250000        7.084170e+06       2.804473e+07       2.15933